# Vision Transformr

**0 前言**<br>
&nbsp;&nbsp;&nbsp;&nbsp;0.1 为什么图像任务也可以使用Transformer？<br>
&nbsp;&nbsp;&nbsp;&nbsp;0.2 从CNN到ViT：视觉建模思路的变化<br>

**1 ViT模型的核心思想与整体结构**<br>
&nbsp;&nbsp;&nbsp;&nbsp;1.1 ViT到底想解决什么问题？<br>
&nbsp;&nbsp;&nbsp;&nbsp;1.2 ViT整体结构与数据流<br>
&nbsp;&nbsp;&nbsp;&nbsp;1.3 ViT和原始Transformer的关系<br>

**2 ViT模型的输入：从图像到Patch序列**<br>
&nbsp;&nbsp;&nbsp;&nbsp;2.1 图像为什么要切成Patch？<br>
&nbsp;&nbsp;&nbsp;&nbsp;2.2 Patch Flatten与Linear Projection<br>
&nbsp;&nbsp;&nbsp;&nbsp;2.3 图像输入在ViT中的张量形状变化<br>

<font color="red">**3 分类标记CLS Token与分类输出**</font><br>
&nbsp;&nbsp;&nbsp;&nbsp;3.1 为什么要额外加入一个分类标记？<br>
&nbsp;&nbsp;&nbsp;&nbsp;3.2 CLS Token如何汇总整张图像的信息？<br>
&nbsp;&nbsp;&nbsp;&nbsp;3.3 分类头MLP Head如何输出类别概率？<br>

**4 ViT中的位置编码**<br>
&nbsp;&nbsp;&nbsp;&nbsp;4.1 图像Patch为什么也需要位置信息？<br>
&nbsp;&nbsp;&nbsp;&nbsp;4.2 ViT中的可学习位置编码<br>
&nbsp;&nbsp;&nbsp;&nbsp;4.3 位置编码与图像二维结构的关系<br>

**5 ViT中的Transformer Encoder主干**<br>
&nbsp;&nbsp;&nbsp;&nbsp;5.1 ViT中的多头自注意力机制<br>
&nbsp;&nbsp;&nbsp;&nbsp;5.2 ViT中的前馈神经网络MLP Block<br>
&nbsp;&nbsp;&nbsp;&nbsp;5.3 残差连接与Layer Normalization<br>

**6 ViT的训练、特点与局限**<br>
&nbsp;&nbsp;&nbsp;&nbsp;6.1 ViT为什么通常需要较大数据量？<br>
&nbsp;&nbsp;&nbsp;&nbsp;6.2 ViT和CNN的归纳偏置差异<br>
&nbsp;&nbsp;&nbsp;&nbsp;6.3 ViT的优势、局限与常见变体<br>


## 3 分类标记CLS Token与分类输出

前一章我们已经看到，一张图像进入ViT之后，会先被切成若干个patch，再经过flatten和线性投影，最终变成一串patch embedding。到这一步为止，模型已经拥有了一整条输入序列。

不过，这时还有一个非常关键的问题没有解决：

**如果这是一项图像分类任务，模型最后应该拿这串序列里的哪一部分去代表“整张图”的信息？**

这个问题看起来简单，实际上非常关键。因为Transformer内部会输出一串更新后的token表示，也就是说，每个patch最后都会得到一个新的向量。但图像分类需要的不是“每个patch分别属于什么类别”，而是“整张图属于什么类别”。所以模型必须想办法把整张图的信息汇总成一个可用于分类的全局表示。

ViT处理这个问题的方式并不是在最后简单地把所有patch向量平均一下，而是采用了一个非常有代表性的设计：**在输入序列最前面额外加入一个特殊的标记，也就是CLS Token。**

这一章我们就专门来讲这个特殊token到底是干什么的，它为什么要存在，以及它最后是怎样参与分类输出的。


### 3.1 为什么要额外加入一个分类标记？

我们先把问题说得更具体一点。

在ViT中，一张图像会先被切成很多patch，然后这些patch会变成一串token序列，例如：

$$
[patch_1, patch_2, patch_3, \ldots, patch_N]
$$

经过Transformer Encoder之后，模型仍然会输出一串同样长度的表示：

$$
[h_1, h_2, h_3, \ldots, h_N]
$$

这里的每个 $h_i$ 都对应一个patch经过多层计算之后的最终表示。

问题在于：**图像分类需要一个“整图级别”的输出，但Encoder天然给出的是“一串patch级别”的输出。那到底该拿哪一个向量去做最后分类？**

如果这个问题不先解决，后面的分类头就无从谈起。

一种最直接的想法是：既然有很多patch向量，那就把它们做平均，或者求和，最后得到一个整体向量。这种思路并不是完全不行，后来的不少模型里也确实会使用全局平均池化这类方法。

但是ViT采用的是另一种更有结构感的办法：**专门在序列里安排一个“负责汇总全局信息的位置”，让模型在计算过程中主动把整张图的重要信息聚合到这个位置上。** 这个特殊位置对应的输入向量，就是`CLS Token`。

也就是说，ViT不是等所有patch都算完以后，再临时想办法把它们压成一个向量；而是从一开始就在输入序列里放进一个“全局代表”，让它和所有patch一起参与后续的注意力计算。

这时序列就不再只是：

$$
[patch_1, patch_2, patch_3, \ldots, patch_N]
$$

而会变成：

$$
[CLS, patch_1, patch_2, patch_3, \ldots, patch_N]
$$

这里最前面的`CLS`，就是额外插入的分类标记。

那么，为什么这种设计是合理的？

关键在于Transformer里的自注意力机制。由于`CLS Token`和其他patch token一起进入Encoder，在每一层注意力计算中，它都可以和整条序列中的其他token发生交互。换句话说，`CLS Token`并不是一个静止不动的占位符，而是会在多层计算中不断吸收来自各个patch的信息，逐渐形成一个更偏向“整张图摘要”的表示。

从这个角度看，`CLS Token`更像是模型专门预留出来的一个“信息汇总站”。它本身一开始并不是某个具体patch，也不对应图像中的某个真实局部区域；它的作用是作为一个全局性的容器，去收集整张图的重要信息。

这里尤其要注意，**CLS Token不是从原始图像里切出来的patch。** 这是初学者最容易误解的点之一。更准确地说：

- 普通patch token来自输入图像的真实局部区域
- CLS Token不是图像中的某个区域
- 它是人为额外加入到序列前面的一个可学习向量

也就是说，CLS Token属于模型设计的一部分，而不是输入图像本身的一部分。

这时你可能会问：**既然CLS Token一开始不是来自图像内容，那它怎么可能代表整张图？**

答案就在“训练”和“注意力交互”这两个词里。虽然CLS Token一开始只是一个可学习向量，但在训练过程中，它会和所有patch token一起被不断更新；而在每一层自注意力中，它又有机会从其他patch那里收集信息。久而久之，模型就会学会把“与分类最相关的全局信息”逐步压到这个特殊位置上。

你可以把它想象成班级里专门负责做总结的同学。其他同学分别掌握了不同片段的信息，而这个“总结者”会不断听取别人发言，最后形成一个更全面的结论。CLS Token在ViT里的角色，就有点类似这种“全局汇总位”。

从工程实现上看，引入CLS Token还有一个直接好处：**模型最后做分类时，就可以统一地取序列最前面那个位置的输出向量，而不必临时再设计额外的汇聚规则。**

这使得整个分类流程更整齐：

1. 在输入序列前面加一个CLS Token。
2. 让它和所有patch一起经过Transformer Encoder。
3. 取最终的CLS输出向量送入分类头。

这样一来，“谁来代表整张图”这个问题，就被转换成了一个非常清晰的结构设计问题，而不是留到最后再临时处理。

到这里，我们已经回答了“为什么要额外加入一个分类标记”这个问题：

- 因为图像分类需要一个整图级别的全局表示
- 而Transformer Encoder天然输出的是一串patch级别表示
- 所以需要人为加入一个专门负责汇总全局信息的位置
- 这个位置就是CLS Token

不过，知道“为什么要有它”还不够。下一节我们还要更进一步看：**CLS Token到底是怎样在注意力计算中逐步汇总整张图信息的？** 也就是说，我们要从“设计动机”进一步走向“具体工作机制”。
